<a href="https://colab.research.google.com/github/shwethashaji-ai/AI-powered-research-assistant/blob/main/AI_powered_research_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -q -U google-genai ipywidgets markdown

In [3]:
import os, getpass
key = None
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not key:
    key = getpass.getpass("Paste your Gemini API key (from aistudio.google.com): ")
os.environ["GEMINI_API_KEY"] = key.strip().strip('"').strip("'")
print("Key loaded." if os.environ["GEMINI_API_KEY"] else "No key found.")

Key loaded.


In [4]:
import io, re, json, time, base64, datetime
import pandas as pd
from google import genai
from google.genai import types
from IPython.display import display, Markdown, HTML, clear_output

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"      # change here if Google AI Studio lists a different free model
RATE_DELAY = 3                  # seconds to pause before each call (free tier is rate limited)
DEPTH = {"Quick": 4000, "Standard": 8000, "Deep": 16000}   # max output tokens (includes hidden thinking)

SYSTEM = (
    "You are a careful research assistant. Structure answers with short markdown headings and bullets. "
    "Be concrete and concise. Clearly separate well-established findings from contested or uncertain claims. "
    "Never invent citations, URLs, quotes or statistics; if you are unsure of a name, number or reference, say so. "
    "End with a section titled 'Suggested next steps' with 3 specific things to look up or try."
)

MODES = {
    "Overview": "Give a clear briefing: key concepts, current state of knowledge, major debates, and open questions.",
    "Literature review": "Write a literature-review style outline: main schools of thought, landmark works and authors (flag uncertainty), methods used, and gaps.",
    "Compare": "Compare the main options, approaches or positions side by side, with trade-offs and when each fits best.",
    "Critique & methods": "Assess the strength of evidence, typical methodological weaknesses, confounders, and how one would design a stronger study.",
    "Research plan": "Propose a practical research plan: sharpened research questions, hypotheses, data sources, methods, and a step-by-step timeline.",
    "Timeline & history": "Trace how the topic developed: key milestones, turning points, and what drove them.",
}
DOC_TASKS = {
    "Summarize each document": "Summarize each document separately: purpose, method, key findings, limitations.",
    "Synthesize across documents": "Synthesize the documents together: shared themes, points of agreement, disagreements, and an overall conclusion.",
    "Find gaps & contradictions": "Identify gaps, contradictions, weak evidence and unanswered questions across the documents.",
    "Answer my question": "Answer the question using ONLY the documents. Name the document each point comes from and say clearly when the documents do not contain the answer.",
}
CITE_STYLES = ["APA 7", "MLA 9", "IEEE", "Chicago", "Harvard", "BibTeX"]
LEVELS = ["Explain like I'm 12", "Undergraduate", "Graduate", "Expert"]
EXTRACT_FIELDS = ["document", "title", "authors", "year", "research_question", "method", "data", "key_findings", "limitations", "future_work"]

def T(text):
    return types.Part.from_text(text=text)

RETRY_KEYS = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL")


class ResearchAssistant:
    def __init__(self):
        self.history, self.docs, self.log = [], [], []

    # ---------- documents ----------
    def add_text(self, name, text):
        self.docs.append({"name": name, "kind": "text", "data": text})

    def add_file(self, name, raw):
        if name.lower().endswith(".pdf"):
            self.docs.append({"name": name, "kind": "pdf", "data": raw})
        else:
            self.docs.append({"name": name, "kind": "text", "data": raw.decode("utf-8", errors="ignore")[:400000]})

    def upload(self):
        from google.colab import files
        for name, raw in files.upload().items():
            self.add_file(name, raw)
        self.list_docs()

    def list_docs(self):
        if not self.docs:
            print("No documents loaded. Run ra.upload() or ra.add_text(name, text).")
        for i, d in enumerate(self.docs, 1):
            print(f"{i}. {d['name']} ({d['kind']})")

    def clear_docs(self):
        self.docs = []

    def _doc_blocks(self):
        parts = []
        for d in self.docs:
            if d["kind"] == "pdf":
                parts.append(T(f"[Document: {d['name']}]"))
                parts.append(types.Part.from_bytes(data=d["data"], mime_type="application/pdf"))
            else:
                parts.append(T(f"[Document: {d['name']}]\n{d['data']}"))
        return parts

    # ---------- engine ----------
    def _pace(self):
        time.sleep(RATE_DELAY)

    def _grounding_sources(self, grounding):
        seen, out = set(), []
        for gc in grounding:
            web = getattr(gc, "web", None)
            u = getattr(web, "uri", None)
            if u and u not in seen:
                seen.add(u)
                out.append((getattr(web, "title", None) or u, u))
        return out

    def _chat(self, content, system=SYSTEM, depth="Standard", web=False, new=True):
        if new:
            self.history = []
        parts = [T(content)] if isinstance(content, str) else list(content)
        self.history.append(types.Content(role="user", parts=parts))
        cfg = types.GenerateContentConfig(system_instruction=system, max_output_tokens=DEPTH[depth])
        if web:
            cfg.tools = [types.Tool(google_search=types.GoogleSearch())]
        handle = display(Markdown("*Working…*"), display_id=True)
        self._pace()
        text, grounding, finish = "", [], ""
        for attempt in range(4):
            text, grounding, finish = "", [], ""
            try:
                for chunk in client.models.generate_content_stream(model=MODEL, contents=self.history, config=cfg):
                    if chunk.text:
                        text += chunk.text
                        handle.update(Markdown(text))
                    cands = getattr(chunk, "candidates", None) or []
                    if cands:
                        gm = getattr(cands[0], "grounding_metadata", None)
                        if gm and getattr(gm, "grounding_chunks", None):
                            grounding.extend(gm.grounding_chunks)
                        fr = getattr(cands[0], "finish_reason", None)
                        if fr:
                            finish = str(fr)
                break
            except Exception as e:
                if attempt < 3 and any(k in str(e) for k in RETRY_KEYS):
                    wait = 20 * (attempt + 1)
                    handle.update(Markdown(f"*Rate limit reached: retrying in {wait}s…*"))
                    time.sleep(wait)
                    continue
                self.history.pop()
                handle.update(Markdown(f"**Error:** `{e}`"))
                return ""
        if not text:
            self.history.pop()
            handle.update(Markdown("**No text returned** (the model may have blocked the request or ran out of tokens). Try a deeper setting or rephrase."))
            return ""
        src = self._grounding_sources(grounding)
        if src:
            text += "\n\n---\n**Sources**\n" + "\n".join(f"{i}. [{t}]({u})" for i, (t, u) in enumerate(src, 1))
        if "MAX_TOKENS" in finish:
            text += "\n\n*(Hit the length limit: ask a follow-up or choose a deeper setting.)*"
        handle.update(Markdown(text))
        self.history.append(types.Content(role="model", parts=[T(text)]))
        self.log.append(text)
        return text

    def _complete(self, content, system=SYSTEM, max_tokens=8000):
        parts = [T(content)] if isinstance(content, str) else list(content)
        cfg = types.GenerateContentConfig(system_instruction=system, max_output_tokens=max_tokens)
        self._pace()
        for attempt in range(4):
            try:
                r = client.models.generate_content(model=MODEL, contents=[types.Content(role="user", parts=parts)], config=cfg)
                return r.text or ""
            except Exception as e:
                if attempt < 3 and any(k in str(e) for k in RETRY_KEYS):
                    time.sleep(20 * (attempt + 1))
                    continue
                raise

    # ---------- use cases ----------
    def research(self, topic, mode="Overview", depth="Standard", web=False, use_docs=False):
        text = f"Task: {MODES[mode]}\nDepth: {depth}\n\nTopic or question:\n{topic}"
        if web:
            text += "\n\nUse web search for current information and cite your sources."
        content = self._doc_blocks() + [T(text)] if (use_docs and self.docs) else text
        return self._chat(content, depth=depth, web=web)

    def follow_up(self, question, depth="Standard", web=False):
        if not self.history:
            print("Start a thread first with ra.research(...) or another use case.")
            return ""
        return self._chat(question, depth=depth, web=web, new=False)

    def docs_task(self, task="Summarize each document", question="", depth="Standard"):
        if not self.docs:
            print("No documents loaded. Run ra.upload() or ra.add_text(name, text).")
            return ""
        text = DOC_TASKS[task] + (f"\n\nQuestion: {question}" if question else "")
        return self._chat(self._doc_blocks() + [T(text)], depth=depth)

    def ask_docs(self, question, depth="Standard"):
        return self.docs_task("Answer my question", question, depth)

    def extract(self, text=None):
        if text:
            content = [T("Document text:\n" + text)]
        elif self.docs:
            content = self._doc_blocks()
        else:
            print("Provide text or load documents first.")
            return None
        content.append(T(
            "For each document, extract these fields: " + ", ".join(EXTRACT_FIELDS) + ". "
            "Return ONLY a JSON list with one object per document, using exactly those keys. "
            "Use null when the document does not say; do not guess."))
        raw = self._complete(content, max_tokens=8000)
        raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
        try:
            df = pd.DataFrame(json.loads(raw))
            display(df)
            return df
        except Exception:
            print("Could not parse JSON; raw output:\n", raw)
            return None

    def cite(self, references, style="APA 7"):
        out = self._complete(
            f"Format each reference below in {style}. Return only the formatted list, one per line "
            f"(separate BibTeX entries with a blank line). If a field is missing, write [missing: field] "
            f"instead of guessing.\n\n{references}",
            system="You are a precise citation formatter. Never invent authors, years, DOIs, volumes or page numbers.",
            max_tokens=4000)
        display(Markdown(out))
        self.log.append(out)
        return out

    def brainstorm(self, topic, n=8, depth="Standard"):
        return self._chat(
            f"Brainstorm {n} distinct, researchable questions about: {topic}\n"
            "For each give: the question, a testable hypothesis, a suitable method, the data needed, and a "
            "feasibility rating (low/medium/high) with one line of reasoning. Finish by ranking the top 3.",
            depth=depth)

    def peer_review(self, draft, depth="Standard"):
        return self._chat(
            "Act as a rigorous but constructive peer reviewer. Review the draft below. Cover: summary, "
            "strengths, major concerns (argument, evidence, method, statistics), minor issues (clarity, "
            "structure, style), and a prioritized list of revisions.\n\nDRAFT:\n" + draft,
            depth=depth)

    def explain(self, concept, level="Undergraduate", depth="Quick"):
        return self._chat(
            f"Explain this at the level '{level}': {concept}\nUse one analogy or example and finish with 3 "
            "check-your-understanding questions.", depth=depth)

    def export(self, name="research-notes"):
        body = "\n\n---\n\n".join(self.log) or "(no notes yet)"
        paths = []
        with open(f"{name}.md", "w") as f:
            f.write(body)
        paths.append(f"{name}.md")
        try:
            import markdown
            html = markdown.markdown(body, extensions=["tables", "fenced_code"])
            with open(f"{name}.html", "w") as f:
                f.write("<meta charset='utf-8'><body style='font-family:Georgia,serif;max-width:760px;margin:2rem auto;line-height:1.6'>"
                        + html + "</body>")
            paths.append(f"{name}.html")
        except Exception:
            pass
        try:
            from google.colab import files
            for p in paths:
                files.download(p)
        except Exception:
            print("Saved:", ", ".join(paths))
        return paths


ra = ResearchAssistant()
print("Assistant ready: use the `ra` object.")

Assistant ready: use the `ra` object.


In [5]:
_ = ra.explain("photosynthesis", "Explain like I'm 12")

**Error:** `404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}`

In [6]:
import ipywidgets as w

def bind(button, out, fn):
    def cb(_):
        with out:
            clear_output()
            try:
                fn()
            except Exception as e:
                print("Error:", e)
    button.on_click(cb)

def TA(ph, h="90px"):
    return w.Textarea(placeholder=ph, layout=w.Layout(width="98%", height=h))

# --- Research ---
r_topic = TA("Topic or question…")
r_mode = w.Dropdown(options=list(MODES), description="Mode")
r_depth = w.Dropdown(options=list(DEPTH), value="Standard", description="Depth")
r_web = w.Checkbox(description="Live web search")
r_docs = w.Checkbox(description="Use loaded docs")
r_go = w.Button(description="Research (new thread)", button_style="primary")
r_fu = w.Text(placeholder="Follow-up question…", layout=w.Layout(width="70%"))
r_fub = w.Button(description="Ask follow-up")
r_out = w.Output()
bind(r_go, r_out, lambda: ra.research(r_topic.value, r_mode.value, r_depth.value, r_web.value, r_docs.value))
bind(r_fub, r_out, lambda: ra.follow_up(r_fu.value, r_depth.value, r_web.value))
t_research = w.VBox([r_topic, w.HBox([r_mode, r_depth]), w.HBox([r_web, r_docs, r_go]), w.HBox([r_fu, r_fub]), r_out])

# --- Documents ---
d_task = w.Dropdown(options=list(DOC_TASKS), description="Task")
d_q = w.Text(placeholder="Question (for 'Answer my question')", layout=w.Layout(width="90%"))
d_list = w.Button(description="List docs")
d_clear = w.Button(description="Clear docs")
d_go = w.Button(description="Run on documents", button_style="primary")
d_x = w.Button(description="Extract to table")
d_out = w.Output()
bind(d_list, d_out, ra.list_docs)
bind(d_clear, d_out, lambda: (ra.clear_docs(), print("Cleared.")))
bind(d_go, d_out, lambda: ra.docs_task(d_task.value, d_q.value))
bind(d_x, d_out, ra.extract)
t_docs = w.VBox([d_task, d_q, w.HBox([d_list, d_clear, d_go, d_x]), d_out])

# --- Citations ---
c_in = TA("One reference per line (even rough: author, title, year, journal…)", "110px")
c_style = w.Dropdown(options=CITE_STYLES, description="Style")
c_go = w.Button(description="Format", button_style="primary")
c_out = w.Output()
bind(c_go, c_out, lambda: ra.cite(c_in.value, c_style.value))
t_cite = w.VBox([c_in, w.HBox([c_style, c_go]), c_out])

# --- Brainstorm ---
b_in = TA("Research area…", "70px")
b_n = w.IntSlider(value=8, min=3, max=15, description="Ideas")
b_go = w.Button(description="Brainstorm", button_style="primary")
b_out = w.Output()
bind(b_go, b_out, lambda: ra.brainstorm(b_in.value, b_n.value))
t_brain = w.VBox([b_in, w.HBox([b_n, b_go]), b_out])

# --- Peer review ---
p_in = TA("Paste your draft here…", "160px")
p_go = w.Button(description="Review", button_style="primary")
p_out = w.Output()
bind(p_go, p_out, lambda: ra.peer_review(p_in.value))
t_peer = w.VBox([p_in, p_go, p_out])

# --- Explain ---
e_in = TA("Concept to explain…", "60px")
e_lvl = w.Dropdown(options=LEVELS, value="Undergraduate", description="Level")
e_go = w.Button(description="Explain", button_style="primary")
e_out = w.Output()
bind(e_go, e_out, lambda: ra.explain(e_in.value, e_lvl.value))
t_explain = w.VBox([e_in, w.HBox([e_lvl, e_go]), e_out])

# --- Export ---
x_go = w.Button(description="Download notes (.md + .html)", button_style="success")
x_out = w.Output()
bind(x_go, x_out, ra.export)
t_export = w.VBox([x_go, x_out])

tabs = w.Tab(children=[t_research, t_docs, t_cite, t_brain, t_peer, t_explain, t_export])
for i, n in enumerate(["Research", "Documents", "Citations", "Brainstorm", "Peer review", "Explain", "Export"]):
    tabs.set_title(i, n)
display(tabs)

In [7]:
MODEL = "gemini-3.8-flash"
_ = ra.explain("photosynthesis", "Explain like I'm 12")

*Working…*

### What is photosynthesis?

Photosynthesis is the process green plants, algae, and some bacteria use to make their own food. 

* The word comes from Greek: **photo** means "light," and **synthesis** means "putting together."
* Instead of eating meals like animals do, plants use light energy to build sugars from basic ingredients found in air and soil.

---

### The solar-powered bakery analogy

Imagine a leaf is a tiny, **solar-powered bakery**:

* **The power source:** Sunlight streaming onto the roof.
* **The chef:** A microscopic green kitchen inside the plant cell called a **chloroplast**. The green pigment inside it, called **chlorophyll**, traps the sunlight like a solar panel.
* **The raw ingredients:** Water drawn up from the soil through the roots, and carbon dioxide ($CO_2$) gas taken in from the air through tiny leaf pores.
* **The finished treat:** **Glucose** (a type of sugar). The plant eats this sugar immediately for energy or packs it away to grow bigger.
* **The kitchen scraps:** **Oxygen**. The bakery doesn't need it, so it vents it out into the air for humans and animals to breathe.

---

### The plant's recipe

* **What goes in (Inputs):**
  * **Sunlight:** Energy to run the process.
  * **Water ($H_2O$):** Absorbed by roots.
  * **Carbon dioxide ($CO_2$):** Absorbed from the air.
* **What comes out (Outputs):**
  * **Glucose ($C_6H_{12}O_6$):** Food and building material for the plant.
  * **Oxygen ($O_2$):** Released into the atmosphere.

---

### Established facts vs. active research

* **Well-established findings:**
  * Plants require light, water, and carbon dioxide to produce glucose and oxygen.
  * Chloroplasts and chlorophyll are the physical machinery responsible for capturing light.
  * Photosynthesis produces most of the oxygen in Earth's atmosphere.

* **Active research areas:**
  * **Artificial photosynthesis:** Scientists are trying to build man-made systems that copy plants to produce clean hydrogen fuel, but these are currently difficult and expensive to scale.
  * **Crop engineering:** Researchers are working to genetically tweak the main plant enzyme (called *RuBisCO*) to make crops waste less energy in hot, dry conditions.

---

### Check your understanding

1. What are the two main "waste products" or outputs of photosynthesis, and which one do humans rely on to breathe?
2. Why do most plant leaves look green instead of blue or red?
3. What would happen to a plant's ability to make food if you sealed it inside a clear glass jar that had all the carbon dioxide removed?

---

### Suggested next steps

1. **Try an experiment:** Place a fresh spinach leaf underwater in a clear glass under a bright lamp, and look for tiny oxygen bubbles forming on the surface after 20 minutes.
2. **Look up an image:** Search for a diagram of a **stoma** (plural: *stomata*) to see what the microscopic "mouths" on leaves look like under a microscope.
3. **Explore deeper:** Look up the difference between the **light-dependent reactions** and the **Calvin cycle** to see the two main stages of how plants process this energy.

In [8]:
RUN_WEB_TEST = True

SAMPLE_A = """Title: Four-Day Work Weeks and Employee Output (sample, fictional)
Authors: A. Rao, B. Mehta. Year: 2023.
Research question: Does a four-day work week change self-reported productivity?
Method: 12-week pilot across 14 mid-size firms, survey at weeks 0, 6 and 12.
Data: 1,150 employees; self-reported productivity on a 1-10 scale.
Findings: Mean productivity rose from 6.4 to 7.1 (p < 0.05). Burnout scores fell by 18%.
Limitations: No control group; self-reported outcomes; volunteer firms only.
Future work: Randomized trials with objective output measures."""

SAMPLE_B = """Title: Remote Work, Output and Collaboration (sample, fictional)
Authors: C. Iyer, D. Gomez. Year: 2021.
Research question: How does remote work affect objective team output?
Method: Quasi-experiment comparing 40 remote and 40 on-site teams over 9 months.
Data: Ticket completion counts and peer-collaboration network metrics.
Findings: Output was unchanged (difference not significant); cross-team collaboration declined by 12%.
Limitations: Teams not randomized; single industry (software).
Future work: Study hybrid schedules and onboarding effects."""

REFS = """Vaswani, Shazeer, Parmar et al. Attention is all you need. NeurIPS 2017
LeCun, Bengio, Hinton. Deep learning. Nature 521, 2015, pages 436-444"""

DRAFT = """Abstract. We show that our new model is better than all baselines. We tested it on one dataset
and it achieved 92% accuracy, so it will generalize to every domain. Future work is unnecessary."""

ra.clear_docs()
ra.add_text("sample_A.txt", SAMPLE_A)
ra.add_text("sample_B.txt", SAMPLE_B)

tests = []
for m in MODES:
    tests.append((f"Research mode: {m}", lambda m=m: ra.research("Remote work and productivity", m, "Quick")))
tests += [
    ("Follow-up", lambda: ra.follow_up("Which finding is the least certain, and why?", "Quick")),
    ("Research using loaded docs", lambda: ra.research("What do these studies suggest?", "Overview", "Quick", use_docs=True)),
    ("Docs: summarize", lambda: ra.docs_task("Summarize each document", depth="Quick")),
    ("Docs: synthesize", lambda: ra.docs_task("Synthesize across documents", depth="Quick")),
    ("Docs: gaps", lambda: ra.docs_task("Find gaps & contradictions", depth="Quick")),
    ("Docs: Q&A", lambda: ra.ask_docs("Which study found no change in output, and what was its limitation?", "Quick")),
    ("Structured extraction", lambda: ra.extract()),
]
for s in CITE_STYLES:
    tests.append((f"Citation: {s}", lambda s=s: ra.cite(REFS, s)))
tests += [
    ("Brainstorm", lambda: ra.brainstorm("hybrid work and wellbeing", 4, "Quick")),
    ("Peer review", lambda: ra.peer_review(DRAFT, "Quick")),
]
for lv in LEVELS:
    tests.append((f"Explain: {lv}", lambda lv=lv: ra.explain("statistical significance", lv)))
if RUN_WEB_TEST:
    tests.append(("Live web research", lambda: ra.research("Latest evidence on four-day work weeks", "Overview", "Quick", web=True)))
tests.append(("Export", lambda: ra.export("selftest-notes")))

results = []
for title, fn in tests:
    display(Markdown(f"\n\n## ▶ {title}"))
    try:
        out = fn()
        ok = out is not None and (len(out) > 0)
    except Exception as e:
        print("FAILED:", e)
        ok = False
    results.append({"use case": title, "status": "✅ pass" if ok else "❌ fail"})

display(Markdown("\n\n## Summary"))
display(pd.DataFrame(results))



## ▶ Research mode: Overview

### Key Concepts

*   **Hybrid vs. Fully Remote:** Hybrid combines in-office and remote days; fully remote operates entirely off-site. Research treats these as distinct models with different outcomes.
*   **Input vs. Output Productivity:** Measuring productivity by hours logged or activities tracked (inputs) versus deliverables completed, quality, and revenue generated (outputs).
*   **Task Type Differentiation:** Deep-focus, solitary tasks (e.g., coding, writing) versus collaborative, exploratory tasks (e.g., brainstorming, complex negotiation).
*   **Proximity Bias:** The tendency of managers to evaluate in-office workers more favorably than remote peers, regardless of objective output.

---

### Well-Established Findings

*   **Hybrid models preserve productivity:** Across numerous studies, a hybrid schedule (typically 2–3 days in the office) shows either neutral or slightly positive effects on individual productivity and job satisfaction compared to full-time office work.
*   **Commute reclamation:** Workers consistently reallocate a significant portion of eliminated commute time back into work duties, increasing total output via longer working hours, even if hourly productivity remains flat.
*   **Reduced turnover:** Offering remote or hybrid flexibility consistently lowers employee quit rates, reducing recruitment and onboarding costs.
*   **Focused work gains:** Individual, non-collaborative tasks are generally completed faster and with fewer interruptions at home.

---

### Contested Claims and Major Debates

*   **Impact on innovation and breakthrough ideas:** 
    *   *Critique:* Physical separation diminishes spontaneous interactions ("water cooler" moments), weakening cross-silo collaboration and novel idea generation.
    *   *Counterclaim:* Digital coordination tools and deliberate asynchronous communication allow broader, more diverse input than co-located teams.
*   **Fully remote vs. hybrid performance:** 
    *   Some evidence (particularly in data-entry or customer service roles) suggests fully remote workers experience a slight productivity drop over time due to isolation, tech friction, or blurred work-life boundaries.
    *   Other studies show sustained or increased output when employees are granted high autonomy.
*   **Measurement discrepancies (Self-report vs. Managerial perception):** 
    *   Employees consistently report higher productivity when working remotely. 
    *   Senior executives often report skepticism, citing invisible slowdowns in alignment, mentoring, and project velocity.

---

### Open Questions

*   **Long-term human capital development:** How does sustained remote work affect the skill acquisition, soft-skill development, and promotion trajectories of junior and entry-level staff?
*   **Aggregate macroeconomic effects:** Does the shift toward remote work increase overall Total Factor Productivity (TFP) at a national level, or does it merely shift costs (e.g., commercial real estate to home office setups)?
*   **Optimal collaboration design:** What is the empirical threshold of in-person time required to sustain team trust and corporate culture without triggering the dissatisfaction associated with rigid office mandates?

---

### Suggested Next Steps

1.  **Examine the WFH Research project:** Review the ongoing survey data and working papers led by researchers Nicholas Bloom (Stanford), Steven Davis (Chicago Booth), and Jose Maria Barrero (ITAM) at [wfhresearch.com](https://wfhresearch.com) for empirical baseline data.
2.  **Audit internal productivity metrics:** Compare team deliverables, cycle times, or objective KPIs across periods of full-office, remote, and hybrid work within your specific context to avoid relying on self-reported surveys.
3.  **Evaluate task-based scheduling:** Test an explicit "task-based" policy (e.g., reserving office days strictly for collaborative workshops, planning, and 1-on-1s, while designating remote days for uninterrupted execution) to measure changes in output.



## ▶ Research mode: Literature review

### Main Schools of Thought

*   **The Optimistic School (Productivity Enhancing):**
    *   *Core thesis:* Remote work boosts productivity by eliminating commutes, offering quiet focus time, reducing absenteeism, and giving workers temporal flexibility.
    *   *Primary focus:* Individual, quantifiable, execution-heavy tasks (e.g., call centers, data entry, routine coding).
*   **The Skeptical School (Productivity Drag & Coordination Costs):**
    *   *Core thesis:* Long-term remote work degrades productivity via communication silos, loss of tacit knowledge transfer, onboarding friction, and reduced spontaneous innovation.
    *   *Primary focus:* Collaborative, creative, and non-routine knowledge work; junior employee development.
*   **The Contingency / Hybrid-Optimal School:**
    *   *Core thesis:* Productivity effects are not binary; they depend heavily on task complexity, firm culture, management style, and cadence. Hybrid models (e.g., 2–3 days in office) capture focus benefits while mitigating coordination penalties.

---

### Landmark Works and Key Authors

*   **Bloom, Liang, Roberts, & Ying (2015)**
    *   *Context:* Pre-pandemic randomized controlled trial (RCT) with Ctrip call center workers in China.
    *   *Finding:* 13% performance increase for home workers, primarily driven by more minutes worked per shift (fewer breaks/sick days) and a quieter working environment.
*   **Choudhury, Foroughi, & Larson (2021)**
    *   *Context:* Transition to "Work from Anywhere" (geographic flexibility) at the US Patent and Trademark Office (USPTO).
    *   *Finding:* 4.4% increase in output with no increase in rework, highlighting gains from worker autonomy and lower cost of living.
*   **Yang et al. (2021, *Nature Human Behaviour*)**
    *   *Context:* Firm-wide natural experiment analyzing communication networks of ~61,000 Microsoft employees during early pandemic WFH.
    *   *Finding:* Communication became more siloed, static, and asynchronous, potentially threatening long-term innovation and cross-functional collaboration.
*   **Gibbs, Mengel, & Siemroth (2023 / working paper 2021)**
    *   *Context:* Study of ~10,000 professionals at an Indian IT services firm.
    *   *Finding:* Total hours worked increased by ~30%, but output fell slightly, resulting in an approximate 20% drop in productivity per hour (driven by communication and coordination overhead).
*   **Barrero, Bloom, & Davis (Ongoing research via WFH Research / SWAA project)**
    *   *Context:* Large-scale monthly survey data on work arrangements and worker self-assessments (US and international).
    *   *Finding:* Documents persistent worker-management expectation gaps and measures the economic value of commute savings. *(Note: rely heavily on self-reported estimates).*

---

### Methodological Approaches

*   **Randomized Controlled Trials (RCTs):** Highly credible for causality (e.g., Bloom et al.), but historically rare and primarily limited to standardized, easily measurable roles.
*   **Natural / Quasi-Experiments:** Leveraging policy shifts, weather shocks, or COVID-19 lockdowns. Useful for large populations, but pandemic-era studies face confounding factors (school closures, health stress).
*   **Passive Telemetry and Administrative Data:** Pulling ERP logs, code commits, emails, or keystrokes. Highly objective on *activity*, but often conflates *busyness* with *high-value output*.
*   **Surveys and Self-Reports:** Flexible and common, but prone to social desirability bias (employees overestimating their remote productivity to retain flexibility).

---

### Well-Established vs. Contested Findings

#### Well-Established Findings
*   **Routine tasks suffer no penalty (and often gain):** Individual, independent, and repetitive tasks perform reliably well in remote environments.
*   **Time-saving dividend:** Commute elimination generates substantial welfare gains; workers routinely allocate a fraction of saved commute time back to employer tasks.
*   **Hybrid parity:** Well-managed hybrid models (2–3 days remote) generally show zero net negative effect on routine short-term productivity compared to full-time office work.

#### Contested and Uncertain Areas
*   **Creativity and Innovation:** Whether asynchronous digital tools can replace the informal "watercooler" interactions necessary for complex creative breakthroughs remains intensely debated and difficult to measure.
*   **Total Output vs. Hours Worked:** Disentangling whether higher output stems from genuine efficiency gains or uncompensated "workday creep" (surplus hours).
*   **Junior Employee Human Capital:** Long-term effects on skill acquisition, tacit learning, and career velocity for early-career hires remain largely unproven due to limited longitudinal post-pandemic data.

---

### Gaps in the Literature

*   **Long-Term Longitudinal Studies:** Most high-profile empirical papers measure 6–18 month windows; multi-year impacts on firm-level total factor productivity and survival are sparse.
*   **Measurement of Complex Knowledge Work:** Reliable, objective output metrics for knowledge workers (e.g., product managers, researchers, executives) beyond proxy metrics like message volume or lines of code.
*   **Interaction with Generative AI:** Limited empirical evidence on whether modern AI tools (e.g., Copilot, LLMs) mitigate the communication and onboarding penalties typically observed in distributed teams.

---

### Suggested next steps

1.  **Examine the latest WFH Research data:** Look up the *Survey of Working Arrangements and Attitudes (SWAA)* by Jose Maria Barrero, Nicholas Bloom, and Steven J. Davis for current quarterly trends on productivity perceptions.
2.  **Compare task-level variation:** Review recent working papers on the *National Bureau of Economic Research (NBER)* using the search string `"remote work" AND "productivity"` to compare differences between software engineering, customer support, and managerial tasks.
3.  **Evaluate firm telemetry metrics:** Investigate how industrial-organizational psychology literature evaluates digital proxies for output (e.g., GitHub PR review latency, Jira cycle times) versus self-reported performance ratings.



## ▶ Research mode: Compare

### Overview of Working Models

Organizations and researchers generally evaluate remote work productivity across three primary models: Full Remote, Hybrid, and Full In-Office. 

---

### Side-by-Side Comparison

| Model | Primary Productivity Drivers | Main Productivity Drains | When It Fits Best |
| :--- | :--- | :--- | :--- |
| **Full Remote** | • Reduced commute fatigue<br>• Fewer in-office interruptions<br>• Flexible scheduling around peak energy hours | • Communication friction (async delays)<br>• Home distractions and isolation<br>• Harder onboarding and junior training | • Geographically dispersed teams<br>• Autonomous, task-driven roles (e.g., software engineering, writing)<br>• Companies with mature async documentation |
| **Hybrid** | • Balance of deep work at home and collaborative work in office<br>• Preserves face-to-face rapport without full-time commute burden | • Coordination complexity ("coordination tax")<br>• Risk of commuting to the office just to sit on video calls | • Knowledge work requiring both complex collaboration and heads-down focus<br>• Teams located near a central hub |
| **Full In-Office** | • Immediate, synchronous alignment<br>• Incidental communication ("watercooler" effects)<br>• Clear boundary between work and personal life | • Commute time and fatigue<br>• Frequent environmental distractions<br>• Reduced autonomy leading to lower morale | • Highly interdependent, fast-pivoting teams<br>• Physical-dependent workflows or strict regulatory/security needs<br>• Early-stage apprentice models |

---

### Well-Established Findings

Research across management literature (e.g., studies led by Nicholas Bloom and peer-reviewed work in organizational behavior) generally confirms the following:

* **Individual focus work increases at home:** Solo, self-contained tasks (such as coding, data analysis, and reading) typically see efficiency gains due to reduced office interruptions.
* **Commute time recovery:** Eliminating commutes directly recovers time, a portion of which employees routinely repurpose into work hours.
* **Onboarding and junior mentorship struggle remotely:** New hires and less experienced workers consistently acquire tacit organizational knowledge more slowly when fully remote compared to co-located environments.
* **Meeting fatigue:** Shifting all communication to video calls increases cognitive load compared to natural in-person conversation.

---

### Contested and Uncertain Claims

The research remains split or inconclusive on several key dynamics:

* **Net macro-productivity impact:** Some empirical studies (notably Bloom’s study of call center workers) show measurable productivity increases, while other recent studies (such as analysis of IT professionals during the pandemic) report output declines when measured by total hours worked versus output produced.
* **Impact on breakthrough innovation:** It is widely theorized that in-person "collisions" drive creative breakthroughs, but empirical proof directly tying office attendance to higher patent quality or breakthrough commercial ideas remains contested.
* **Employee self-reporting vs. manager assessment:** A persistent gap exists where employees report being significantly more productive at home, while managers express skepticism, often due to a reliance on "presence" as a proxy for performance.
* **Long-term cultural degradation:** Whether fully remote organizations suffer gradual, long-term erosion of trust and social capital—or if this can be fully mitigated by strong intentional culture—has not yet been studied across full multi-decade business cycles.

---

### Suggested Next Steps

1. **Audit task types across your team:** Categorize weekly workloads into "heads-down individual tasks" versus "high-bandwidth collaborative tasks" to determine your team's baseline need for co-location.
2. **Review empirical research on hybrid schedules:** Search for the 2022–2024 hybrid work studies by Nicholas Bloom (Stanford University) and the National Bureau of Economic Research (NBER) to evaluate data on 2–3 day hybrid arrangements.
3. **Run a structured async trial:** Test a two-week period limiting all status updates to shared documents or project boards instead of meetings to measure changes in task completion speed.



## ▶ Research mode: Critique & methods

*Rate limit reached: retrying in 40s…*

KeyboardInterrupt: 

In [9]:
def chat():
    mode, depth, web = "Overview", "Standard", False
    started = False
    print("Type a question and press Enter. The first question starts a thread; later ones are follow-ups.")
    print("Commands: /new  /mode NAME  /depth NAME  /web on|off  /docs  /export  /help  /quit\n")
    while True:
        try:
            q = input(f"You [{mode} | {depth} | web {'on' if web else 'off'}]: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nStopped.")
            break
        if not q:
            continue
        low = q.lower()
        if low in ("/quit", "/exit", "quit", "exit"):
            print("Bye.")
            break
        if low == "/help":
            print("Modes:", ", ".join(MODES), "\nDepths:", ", ".join(DEPTH))
            continue
        if low == "/new":
            started = False
            print("New thread: your next question starts fresh.")
            continue
        if low == "/docs":
            ra.list_docs()
            continue
        if low == "/export":
            ra.export()
            continue
        if low.startswith("/web"):
            web = low.endswith("on")
            print("Web search:", "on" if web else "off")
            continue
        if low.startswith("/mode") or low.startswith("/depth"):
            cmd, _, arg = q.partition(" ")
            options = MODES if cmd.lower() == "/mode" else DEPTH
            match = [k for k in options if k.lower().startswith(arg.strip().lower())] if arg.strip() else []
            if match:
                if cmd.lower() == "/mode":
                    mode = match[0]
                else:
                    depth = match[0]
                print("Set to:", match[0])
            else:
                print("Options:", ", ".join(options))
            continue
        if not started:
            out = ra.research(q, mode, depth, web, use_docs=bool(ra.docs))
            started = bool(out)
        else:
            ra.follow_up(q, depth, web)
        print()

chat()

Type a question and press Enter. The first question starts a thread; later ones are follow-ups.
Commands: /new  /mode NAME  /depth NAME  /web on|off  /docs  /export  /help  /quit

You [Overview | Standard | web off]: what is temperature 


### Key concepts

* **Thermodynamic definition:** Temperature ($T$) is formally defined as the rate of change of internal energy ($U$) with respect to entropy ($S$) at constant volume and particle number:  
  $$\frac{1}{T} = \left(\frac{\partial S}{\partial U}\right)_{V, N}$$
* **Kinetic theory definition:** In classical mechanics and ideal gases, temperature is proportional to the average translational kinetic energy of the constituent particles ($\langle E_k \rangle = \frac{3}{2} k_B T$).
* **Thermal equilibrium (Zeroth Law):** Two systems placed in thermal contact will exchange energy until they reach thermal equilibrium; they are defined as having the same temperature when net heat flow ceases.
* **Absolute scale:** The Kelvin scale sets its baseline at absolute zero ($0\text{ K}$), the theoretical state where classical particle motion ceases and a system is in its quantum ground state.

---

### Current state of knowledge (well-established)

* **SI standard definition:** Since the 2019 redefinition of SI units, the kelvin is fundamentally defined by fixing the numerical value of the Boltzmann constant ($k_B$) to exactly $1.380649 \times 10^{-23}\text{ J}\cdot\text{K}^{-1}$.
* **Heat flow direction:** In isolated systems, thermal energy spontaneously flows from a body of higher temperature to one of lower temperature, consistent with the Second Law of Thermodynamics (entropy maximization).
* **Statistical state occupancy:** Temperature determines the relative population of quantum states via the Boltzmann factor ($e^{-E / k_B T}$). At higher temperatures, higher-energy microstates become more accessible.
* **Zero-point energy:** Reaching absolute zero does not mean all motion stops; quantum systems retain a non-zero minimum energy (zero-point energy) dictated by the Heisenberg uncertainty principle.

---

### Debates and edge cases

* **Negative absolute temperatures:**
  * *Established behavior:* Systems with an upper bound on their energy states (such as localized nuclear spins or cold atoms in optical lattices) can exhibit population inversion, causing entropy to decrease as energy increases ($\partial S / \partial U < 0$).
  * *Debate:* Theoretical physicists debate whether negative absolute temperatures are truly "hotter than infinity" (as suggested by the standard Boltzmann entropy definition) or an artifact of the entropy formulation, with some arguing that the Gibbs volume entropy definition strictly forbids $T < 0$.
* **Relativistic temperature transformation:**
  * There is no universally accepted consensus on how temperature transforms under a Lorentz boost. 
  * Competing formulations include the Planck-Einstein view ($T = T_0 \sqrt{1 - v^2/c^2}$), the Ott view ($T = T_0 / \sqrt{1 - v^2/c^2}$), and Landsberg’s view that temperature should be treated as a Lorentz invariant scalar.
* **Non-equilibrium and small-scale regimes:**
  * Temperature strictly applies only to systems in thermodynamic equilibrium. Defining an "effective temperature" in non-equilibrium states (e.g., glasses, active matter, turbulence) remains an active area of theoretical modeling.

---

### Open questions

* **Thermodynamics of few-particle systems:** At what lower limit of particle number ($N$) does the statistical definition of temperature break down into quantum fluctuations?
* **Planck temperature and quantum gravity:** Standard physics breaks down at the Planck temperature ($\sim 1.4 \times 10^{32}\text{ K}$), leaving open the question of how thermal properties behave at the quantum gravitational scale (such as during the earliest moments of the Big Bang or inside black holes).

---

### Suggested next steps

1. Read a derivation of the thermodynamic definition of temperature using statistical mechanics (e.g., via microcanonical and canonical ensembles) to understand the role of entropy.
2. Review the 2019 SI redefinition literature published by the BIPM (International Bureau of Weights and Measures) regarding the fixation of the Boltzmann constant.
3. Compare the papers on the negative temperature debate (specifically contrasting the Boltzmann and Gibbs entropy perspectives by Dunkel & Hilbert versus Frenkel & Warren).


You [Overview | Standard | web off]: /quit
Bye.


In [10]:
print("Model:", MODEL)
try:
    r = client.models.generate_content(model=MODEL, contents="Say hello in 5 words")
    print("Reply:", r.text)
except Exception as e:
    print("Error:", e)

Model: gemini-3.8-flash
Reply: Hello there, how are you?
